# Jupiter JWST Data Finder, False-Color Processor, and Full HD Video Maker

This notebook builds a complete Jupiter image/video workflow:

1. Search public JWST Jupiter observations from the MAST archive.
2. Download science FITS products, preferring image-ready or cube-like products.
3. Read FITS image/cube data and extract one or more usable Jupiter images.
4. Create a false-color Jupiter image:
   - **multi-filter false color** when multiple filters/products are available;
   - **single-band pseudo false color** as a fallback.
5. Export:
   - `jupiter_false_color.png`
   - `jupiter_false_color_fhd.png` at 1920 × 1080
   - `jupiter_false_color_fhd_clip.mp4` at 1920 × 1080
   - optional `jupiter_frame_sequence_fhd.mp4` when a true frame cube is found.

Robustness notes:

- This version avoids the Astropy `TableMergeError` caused by stacking MAST product rows with mixed metadata types, such as `productGroupDescription`.
- For moving Solar System targets, it tries MAST `target_name` searches first and only falls back to sky-coordinate search when needed.
- The output video is forced to Full HD, independent of the original FITS image size.



## 1. Install or upgrade packages

Run this cell once in a fresh environment. In hosted notebooks, restart the kernel after installation if imports fail.


In [ ]:

# Uncomment the next line in a fresh Python/Jupyter environment.
# %pip install -U numpy astropy astroquery imageio imageio-ffmpeg pillow tqdm matplotlib



## 2. Imports


In [ ]:

from __future__ import annotations

from pathlib import Path
import io
import math
import re
import warnings
from dataclasses import dataclass
from typing import Any, Iterable

import numpy as np
from astropy.io import fits
from astropy.table import Table
from astroquery.mast import Observations

from PIL import Image, ImageDraw, ImageFont
import imageio.v2 as imageio
from tqdm.auto import tqdm

from IPython.display import display, Video, Image as IPyImage
import matplotlib.pyplot as plt

print("Imports OK")


## 3. Configuration

The defaults below search Jupiter in public JWST/NIRCam data using `PROGRAM_ID = None`, so the notebook searches the archive by target name first. Adjust the values if you want a different instrument, radius, search limits, or video settings.


In [ ]:
# ---------------------------
# Target and archive search
# ---------------------------
TARGET_NAME = "Jupiter"
OBS_COLLECTION = "JWST"
INSTRUMENT = "NIRCAM"
RADIUS = "0.5 deg"

# For Jupiter, keep PROGRAM_ID=None to search all public JWST observations by archive target name.
# You can set a specific proposal/program ID later if you want to restrict the search.
PROGRAM_ID = None

# Product preferences. I2D is often best for still false-color images;
# CALINTS/RATEINTS can contain integration/frame cubes for video.
PREFER_PRODUCT_KEYS = ("I2D", "CALINTS", "RATEINTS", "CAL", "RATE")

# Download/search limits. Increase if the first products do not contain the filters/frames you want.
MAX_OBS_TO_TRY = 50
MAX_PRODUCTS_TO_TRY_PER_OBS = 8
MAX_PRODUCTS_TO_DOWNLOAD = 10
MAX_FRAMES_TOTAL = 300

# ---------------------------
# Optional local-image mode
# ---------------------------
# Set this to True if you already have a Jupiter image and only want false-color + FHD video.
USE_LOCAL_IMAGE_ONLY = False
LOCAL_IMAGE_PATH = ""  # Example: "/path/to/jupiter.png"

# ---------------------------
# Output and video settings
# ---------------------------
TARGET_SLUG = re.sub(r"[^a-z0-9]+", "_", TARGET_NAME.lower()).strip("_")
OUT_DIR = Path(f"jwst_{TARGET_SLUG}_false_color")
DATA_DIR = OUT_DIR / "data"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

FHD_SIZE = (1920, 1080)
FPS = 24
STILL_CLIP_SECONDS = 8
FRAME_SEQUENCE_FPS = 12
VIDEO_FIT_MODE = "contain"  # "contain" keeps full planet visible; "cover" fills the frame.
VIDEO_ZOOM_START = 1.00
VIDEO_ZOOM_END = 1.08

FALSE_COLOR_PATH = OUT_DIR / f"{TARGET_SLUG}_false_color.png"
FALSE_COLOR_FHD_PATH = OUT_DIR / f"{TARGET_SLUG}_false_color_fhd.png"
STILL_CLIP_PATH = OUT_DIR / f"{TARGET_SLUG}_false_color_fhd_clip.mp4"
FRAME_SEQUENCE_PATH = OUT_DIR / f"{TARGET_SLUG}_frame_sequence_fhd.mp4"

print("Target:", TARGET_NAME)
print("Output folder:", OUT_DIR.resolve())
print("Main video will be:", STILL_CLIP_PATH.resolve())



## 4. Utility functions


In [ ]:

@dataclass
class JupiterProduct:
    fits_path: Path
    filename: str
    image: np.ndarray
    cube: np.ndarray | None
    filter_label: str
    wavelength_um: float | None
    obs_id: str | None = None
    subgroup: str | None = None


def _as_str(value: Any, default: str = "") -> str:
    try:
        if value is None:
            return default
        return str(value)
    except Exception:
        return default


def _table_col(tbl: Table, possible_names: Iterable[str]) -> str | None:
    lower = {c.lower(): c for c in tbl.colnames}
    for name in possible_names:
        if name.lower() in lower:
            return lower[name.lower()]
    return None


def _row_to_one_row_table(row) -> Table:
    return Table({name: [row[name]] for name in row.colnames})


def normalize_target_str(s: str) -> str:
    return (s or "").strip().upper()


def filter_to_target(obs_tbl: Table, target_name: str) -> Table:
    if obs_tbl is None or len(obs_tbl) == 0:
        return obs_tbl
    tgt = normalize_target_str(target_name)
    for col in ("target_name", "target", "targname", "objname"):
        if col in obs_tbl.colnames:
            vals = np.char.upper(obs_tbl[col].astype(str))
            mask = np.char.find(vals, tgt) >= 0
            if np.any(mask):
                return obs_tbl[mask]
    return obs_tbl



def dedupe_table_by_first(tbl: Table, key_candidates=("obsid", "obs_id")) -> Table:
    """Return the first row for each unique observation key without using vstack."""
    if tbl is None or len(tbl) == 0:
        return Table() if tbl is None else tbl
    for key in key_candidates:
        if key in tbl.colnames:
            seen = set()
            keep = []
            for i, value in enumerate(tbl[key]):
                s = _as_str(value)
                if s not in seen:
                    seen.add(s)
                    keep.append(i)
            return tbl[keep]
    return tbl


def query_archive_by_target_name(
    target: str = TARGET_NAME,
    obs_collection: str = OBS_COLLECTION,
) -> Table:
    """
    Search JWST observations by the MAST archive target_name field.

    This is more reliable for moving Solar System targets such as Jupiter
    than a sky-coordinate resolver. It returns the first successful query
    rather than stacking multiple query results, avoiding mixed-type table
    merge errors.
    """
    target_clean = target.strip()
    queries = [
        target_clean,
        target_clean.upper(),
        f"*{target_clean}*",
        f"*{target_clean.upper()}*",
    ]

    last_error = None
    for q in queries:
        try:
            obs = Observations.query_criteria(
                obs_collection=obs_collection,
                target_name=q,
            )
            if obs is not None and len(obs) > 0:
                obs = filter_to_target(obs, target)
                if obs is not None and len(obs) > 0:
                    return dedupe_table_by_first(obs)
        except Exception as exc:
            last_error = exc
            continue

    if last_error is not None:
        print(f"Archive target-name search did not return usable rows. Last error: {type(last_error).__name__}: {last_error}")
    return Table()


def query_jupiter_observations(
    target: str = TARGET_NAME,
    instrument: str = INSTRUMENT,
    radius: str = RADIUS,
    program_id: int | None = PROGRAM_ID,
    obs_collection: str = OBS_COLLECTION,
) -> Table:
    """Query MAST for Jupiter/JWST observations and return a filtered Astropy table."""
    if program_id is None:
        # For Jupiter and other Solar System bodies, try archive target_name first.
        obs = query_archive_by_target_name(target, obs_collection=obs_collection)

        # Fallback for targets that can be resolved to fixed sky coordinates.
        if obs is None or len(obs) == 0:
            try:
                obs = Observations.query_object(target, radius=radius)
            except Exception as exc:
                raise RuntimeError(
                    f'Could not find observations for "{target}". '
                    "MAST target-name search returned no rows, and the coordinate resolver failed. "
                    f"Original resolver error: {type(exc).__name__}: {exc}"
                )
    else:
        obs = Observations.query_criteria(obs_collection=obs_collection, proposal_id=str(program_id))
        obs = filter_to_target(obs, target)

    if obs is None or len(obs) == 0:
        return Table()

    if "obs_collection" in obs.colnames:
        obs = obs[np.char.upper(obs["obs_collection"].astype(str)) == obs_collection.upper()]

    obs = filter_to_target(obs, target)

    if instrument and "instrument_name" in obs.colnames:
        inst = np.char.upper(obs["instrument_name"].astype(str))
        obs = obs[np.char.find(inst, instrument.upper()) >= 0]

    # Prefer science observations when available.
    if "intentType" in obs.colnames:
        intent = np.char.upper(obs["intentType"].astype(str))
        sci = obs[intent == "SCIENCE"]
        if len(sci) > 0:
            obs = sci

    # Prefer longer/denser observations if available.
    for sort_col in ("t_exptime", "t_min"):
        if sort_col in obs.colnames and len(obs) > 0:
            obs.sort(sort_col)
            obs = obs[::-1]
            break

    return dedupe_table_by_first(obs)


def preview_table(tbl: Table, max_rows: int = 8):
    if tbl is None or len(tbl) == 0:
        print("No rows to preview.")
        return
    cols = [c for c in [
        "obs_id", "target_name", "instrument_name", "filters", "proposal_id", "t_exptime", "t_min", "t_max"
    ] if c in tbl.colnames]
    if cols:
        display(tbl[:max_rows][cols])
    else:
        display(tbl[:max_rows])


def subgroup_col(tbl: Table) -> str | None:
    return _table_col(tbl, ("productSubGroupDescription", "productSubGroupDesc", "productsubgroupdescription"))


def product_priority(row) -> tuple[int, int, str]:
    """Lower is better. Prioritize science-ready products and likely image/cube products."""
    filename = _as_str(row["productFilename"] if "productFilename" in row.colnames else "").lower()
    subgroup = ""
    sg_col = subgroup_col(row._table) if hasattr(row, "_table") else None
    if sg_col:
        subgroup = _as_str(row[sg_col]).upper()

    priority_map = {"I2D": 0, "CALINTS": 1, "RATEINTS": 2, "CAL": 3, "RATE": 4}
    rank = 99
    for key, val in priority_map.items():
        if subgroup == key or f"_{key.lower()}" in filename or key.lower() in filename:
            rank = min(rank, val)

    # Prefer files that look less auxiliary.
    aux_penalty = 0
    bad_terms = ("uncal", "traps", "segm", "cat", "asn", "crf", "thumb")
    if any(term in filename for term in bad_terms):
        aux_penalty = 10

    return (rank + aux_penalty, len(filename), filename)


def pick_candidate_products_for_obs(obs_row) -> Table:
    products = Observations.get_product_list(obs_row)
    if products is None or len(products) == 0:
        return Table()

    try:
        products = Observations.filter_products(products, productType="SCIENCE", extension="fits")
    except Exception:
        # Fall back to manual FITS filtering if filter_products changes or fails.
        if "productFilename" in products.colnames:
            fn = np.char.lower(products["productFilename"].astype(str))
            products = products[np.char.endswith(fn, ".fits") | np.char.endswith(fn, ".fits.gz")]

    if products is None or len(products) == 0:
        return Table()

    # IMPORTANT: do not rebuild the table with vstack(row tables).
    # MAST product tables can contain mixed metadata types in columns such as
    # productGroupDescription, e.g. masked ints in most rows and a string in one row.
    # vstack tries to merge dtypes and can raise TableMergeError.
    # Sorting by row index preserves the original MAST table dtypes/masks.
    order = sorted(range(len(products)), key=lambda i: product_priority(products[i]))
    return products[order]


def download_one_product(product_row, download_dir: Path = DATA_DIR) -> Path:
    one = _row_to_one_row_table(product_row)
    manifest = Observations.download_products(one, download_dir=str(download_dir), cache=True)
    if manifest is None or len(manifest) == 0:
        raise RuntimeError("MAST returned an empty download manifest.")
    local_col = _table_col(manifest, ("Local Path", "local_path", "LocalPath"))
    if local_col is None:
        raise RuntimeError(f"Could not find local path in manifest columns: {manifest.colnames}")
    return Path(manifest[local_col][0])


def to_frame_cube(arr: np.ndarray) -> np.ndarray:
    """Convert common FITS SCI array layouts into a frame cube shaped (n, y, x)."""
    a = np.asarray(arr)
    if a.ndim == 2:
        return a[None, :, :]
    if a.ndim == 3:
        # Most JWST integration cubes are (nint, y, x). If the last axis looks frame-like, move it.
        if a.shape[0] <= max(a.shape[1], a.shape[2]):
            return a
        if a.shape[2] <= max(a.shape[0], a.shape[1]):
            return np.moveaxis(a, 2, 0)
        raise ValueError(f"Unclear 3D layout: {a.shape}")
    if a.ndim == 4:
        # Common JWST case: (nints, ngroups, y, x). Average groups.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            return np.nanmean(a, axis=1)
    raise ValueError(f"Unsupported FITS data shape: {a.shape}")


def parse_wavelength_um(label: str | None) -> float | None:
    """Parse labels like F212N/F430M/F070W to approximate wavelength in microns."""
    if not label:
        return None
    matches = re.findall(r"F(\d{3,4})[A-Z]", label.upper())
    if not matches:
        return None
    vals = []
    for m in matches:
        try:
            vals.append(int(m) / 100.0)
        except Exception:
            pass
    return max(vals) if vals else None


def filter_label_from_header_and_name(headers: list[fits.Header], filename: str) -> str:
    labels = []
    for hdr in headers:
        for key in ("FILTER", "PUPIL", "BAND", "CHANNEL"):
            val = _as_str(hdr.get(key, "")).strip().upper()
            if val and val not in {"CLEAR", "NONE", "N/A", "NULL", "ANY"} and val not in labels:
                labels.append(val)

    # Filename fallback: find filter-like strings.
    for m in re.findall(r"F\d{3,4}[A-Z]", filename.upper()):
        if m not in labels:
            labels.append(m)

    if not labels:
        return "UNKNOWN"
    return "+".join(labels)


def read_fits_product(fits_path: Path) -> tuple[np.ndarray, np.ndarray, str, float | None]:
    """Read FITS SCI data and return (summary_image, cube, filter_label, wavelength_um)."""
    with fits.open(fits_path, memmap=False) as hdul:
        hdu_names = [h.name for h in hdul]
        if "SCI" in hdul:
            sci_hdu = hdul["SCI"]
        else:
            # Fallback to the first HDU with array data.
            sci_hdu = next((h for h in hdul if getattr(h, "data", None) is not None), None)
            if sci_hdu is None:
                raise RuntimeError(f"No image data found. HDUs={hdu_names}")

        data = sci_hdu.data
        if data is None:
            raise RuntimeError(f"Selected HDU has no data. HDUs={hdu_names}")

        cube = to_frame_cube(np.asarray(data, dtype=np.float32))
        headers = [hdul[0].header, sci_hdu.header]
        label = filter_label_from_header_and_name(headers, fits_path.name)

    if cube.shape[0] > MAX_FRAMES_TOTAL:
        idx = np.linspace(0, cube.shape[0] - 1, MAX_FRAMES_TOTAL).round().astype(int)
        cube = cube[idx]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        image = np.nanmedian(cube, axis=0).astype(np.float32)

    wavelength = parse_wavelength_um(label)
    return image, cube.astype(np.float32), label, wavelength


def finite_fill_value(arr: np.ndarray, default: float = 0.0) -> float:
    a = np.asarray(arr)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return float(default)
    return float(np.nanmedian(finite))


def background_subtract(frame: np.ndarray, percentile: float = 10) -> np.ndarray:
    a = np.asarray(frame, dtype=np.float32)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return np.zeros_like(a, dtype=np.float32)
    bg = np.nanpercentile(finite, percentile)
    return np.nan_to_num(a - bg, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def find_disk_bbox(frame: np.ndarray, pad: int = 40, threshold_percentile: float = 85) -> tuple[int, int, int, int]:
    a = background_subtract(frame)
    if a.ndim != 2 or a.size == 0:
        return 0, 0, 0, 0
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        h, w = a.shape
        return 0, h, 0, w
    threshold = np.nanpercentile(finite, threshold_percentile)
    mask = a > threshold
    if not np.any(mask):
        h, w = a.shape
        return 0, h, 0, w
    ys, xs = np.where(mask)
    h, w = a.shape
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(h, int(ys.max()) + pad + 1)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(w, int(xs.max()) + pad + 1)
    if y1 <= y0 or x1 <= x0:
        return 0, h, 0, w
    return y0, y1, x0, x1


def crop_around_signal(img: np.ndarray, pad: int = 50) -> np.ndarray:
    y0, y1, x0, x1 = find_disk_bbox(img, pad=pad)
    return np.asarray(img)[y0:y1, x0:x1]


def robust_limits(arr: np.ndarray, low: float = 1, high: float = 99.7) -> tuple[float, float]:
    finite = np.asarray(arr)[np.isfinite(arr)]
    if finite.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.nanpercentile(finite, [low, high])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = float(np.nanmin(finite)), float(np.nanmax(finite))
    if vmax <= vmin:
        vmax = vmin + 1.0
    return float(vmin), float(vmax)


def scale_to_uint8(img: np.ndarray, low: float = 1, high: float = 99.7, stretch: str = "asinh") -> np.ndarray:
    a = background_subtract(img)
    vmin, vmax = robust_limits(a, low, high)
    x = np.clip((a - vmin) / (vmax - vmin + 1e-12), 0, 1)
    if stretch == "asinh":
        x = np.arcsinh(10 * x) / np.arcsinh(10)
    elif stretch == "sqrt":
        x = np.sqrt(x)
    return (255 * x).astype(np.uint8)


def resize_gray_to_shape(gray: np.ndarray, shape_hw: tuple[int, int]) -> np.ndarray:
    target_h, target_w = shape_hw
    im = Image.fromarray(gray.astype(np.uint8), mode="L")
    im = im.resize((target_w, target_h), resample=Image.Resampling.LANCZOS)
    return np.asarray(im)


def pseudo_false_color_single_band(img: np.ndarray) -> np.ndarray:
    """Create an illustrative RGB false color from one intensity image."""
    base = scale_to_uint8(img)
    x = base.astype(np.float32) / 255.0
    r = np.clip(255 * np.power(x, 0.65), 0, 255).astype(np.uint8)
    g = np.clip(255 * np.power(x, 1.00), 0, 255).astype(np.uint8)
    b = np.clip(255 * np.power(x, 1.80), 0, 255).astype(np.uint8)
    return np.stack([r, g, b], axis=-1)


def make_false_color_from_products(products: list[JupiterProduct]) -> tuple[np.ndarray, str]:
    """Build false-color RGB from multiple filter products, or a pseudo-color fallback."""
    if not products:
        raise RuntimeError("No products available for false-color creation.")

    # Keep products with valid images.
    valid = [p for p in products if p.image is not None and np.asarray(p.image).ndim == 2]
    if not valid:
        raise RuntimeError("No 2D image summaries found in downloaded products.")

    # Sort by wavelength when available; otherwise keep original order.
    valid_sorted = sorted(
        valid,
        key=lambda p: (p.wavelength_um is None, p.wavelength_um if p.wavelength_um is not None else 999, p.filename),
    )

    # Remove near-duplicate filter labels while preserving order.
    unique = []
    seen = set()
    for p in valid_sorted:
        key = p.filter_label
        if key not in seen:
            unique.append(p)
            seen.add(key)
    if len(unique) < 2 and len(valid_sorted) > len(unique):
        unique = valid_sorted[: min(3, len(valid_sorted))]

    if len(unique) >= 3:
        b_src = unique[0]
        g_src = unique[len(unique) // 2]
        r_src = unique[-1]
        mode = f"Multi-filter false color: R={r_src.filter_label}, G={g_src.filter_label}, B={b_src.filter_label}"
        imgs = [crop_around_signal(src.image, pad=60) for src in (r_src, g_src, b_src)]
        channels = [scale_to_uint8(img) for img in imgs]
        target_shape = (max(c.shape[0] for c in channels), max(c.shape[1] for c in channels))
        channels = [resize_gray_to_shape(c, target_shape) for c in channels]
        rgb = np.stack(channels, axis=-1)
        return rgb, mode

    if len(unique) == 2:
        short, long = unique[0], unique[-1]
        mode = f"Two-filter false color: R={long.filter_label}, G=blend, B={short.filter_label}"
        short_img = crop_around_signal(short.image, pad=60)
        long_img = crop_around_signal(long.image, pad=60)
        b = scale_to_uint8(short_img)
        r = scale_to_uint8(long_img)
        target_shape = (max(r.shape[0], b.shape[0]), max(r.shape[1], b.shape[1]))
        r = resize_gray_to_shape(r, target_shape)
        b = resize_gray_to_shape(b, target_shape)
        g = ((r.astype(np.float32) + b.astype(np.float32)) / 2).astype(np.uint8)
        rgb = np.stack([r, g, b], axis=-1)
        return rgb, mode

    # Single-product fallback.
    src = unique[0]
    mode = f"Single-band pseudo false color from {src.filter_label}"
    img = crop_around_signal(src.image, pad=60)
    return pseudo_false_color_single_band(img), mode


def annotate_rgb(rgb: np.ndarray, text: str, height: int = 42) -> np.ndarray:
    im = Image.fromarray(rgb.astype(np.uint8)).convert("RGB")
    draw = ImageDraw.Draw(im)
    draw.rectangle([0, 0, im.size[0], height], fill=(0, 0, 0))
    draw.text((10, 11), text, fill=(255, 255, 255))
    return np.asarray(im)


def save_rgb(rgb: np.ndarray, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(rgb.astype(np.uint8)).save(path)
    return path


def fit_image_to_canvas(rgb: np.ndarray, size: tuple[int, int] = FHD_SIZE, mode: str = "contain") -> np.ndarray:
    """Return an RGB array exactly matching size=(width,height)."""
    W, H = size
    img = Image.fromarray(rgb.astype(np.uint8)).convert("RGB")
    iw, ih = img.size
    if iw <= 0 or ih <= 0:
        raise ValueError("Image has invalid dimensions.")

    if mode == "cover":
        scale = max(W / iw, H / ih)
    else:
        scale = min(W / iw, H / ih)

    nw, nh = max(1, int(round(iw * scale))), max(1, int(round(ih * scale)))
    resized = img.resize((nw, nh), resample=Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (W, H), (0, 0, 0))
    canvas.paste(resized, ((W - nw) // 2, (H - nh) // 2))
    return np.asarray(canvas)


def ease_in_out(t: float) -> float:
    return 0.5 - 0.5 * math.cos(math.pi * t)


def render_fhd_clip_from_image(
    rgb: np.ndarray,
    out_path: Path = STILL_CLIP_PATH,
    seconds: float = STILL_CLIP_SECONDS,
    fps: int = FPS,
    size: tuple[int, int] = FHD_SIZE,
    fit_mode: str = VIDEO_FIT_MODE,
    zoom_start: float = VIDEO_ZOOM_START,
    zoom_end: float = VIDEO_ZOOM_END,
    label: str | None = None,
) -> Path:
    """Render a 1920x1080 Ken-Burns-style MP4 clip from one image."""
    W, H = size
    base = fit_image_to_canvas(rgb, size=size, mode=fit_mode)
    base_img = Image.fromarray(base).convert("RGB")
    nframes = max(1, int(round(seconds * fps)))

    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(
        out_path,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=16,
        ffmpeg_params=["-pix_fmt", "yuv420p", "-movflags", "+faststart"],
    )

    try:
        for i in tqdm(range(nframes), desc=f"Rendering {out_path.name}"):
            t = 0 if nframes == 1 else i / (nframes - 1)
            e = ease_in_out(t)
            zoom = zoom_start + (zoom_end - zoom_start) * e
            zw, zh = int(round(W * zoom)), int(round(H * zoom))
            frame = base_img.resize((zw, zh), resample=Image.Resampling.LANCZOS)

            # Slow diagonal pan within the zoomed frame.
            max_x = max(0, zw - W)
            max_y = max(0, zh - H)
            left = int(round(max_x * (0.35 + 0.30 * e)))
            top = int(round(max_y * (0.40 - 0.20 * e)))
            frame = frame.crop((left, top, left + W, top + H))

            arr = np.asarray(frame)
            if label:
                arr = annotate_rgb(arr, label, height=44)
            writer.append_data(arr)
    finally:
        writer.close()

    return out_path


def render_fhd_video_from_cube(
    cube: np.ndarray,
    out_path: Path = FRAME_SEQUENCE_PATH,
    fps: int = FRAME_SEQUENCE_FPS,
    size: tuple[int, int] = FHD_SIZE,
    label_prefix: str = "Jupiter frame sequence",
) -> Path:
    """Render a grayscale frame cube as a 1920x1080 MP4."""
    if cube is None or np.asarray(cube).ndim != 3 or len(cube) < 2:
        raise ValueError("Need a 3D cube with at least two frames.")

    processed = np.asarray([crop_around_signal(f, pad=60) for f in cube], dtype=object)
    # Use uncropped cube if object-array crop shapes differ too much.
    if processed.dtype == object:
        processed = [background_subtract(f) for f in cube]
    else:
        processed = list(processed)

    # Shared contrast from a sample for smoother video.
    sample = np.concatenate([np.asarray(f).ravel()[::20] for f in processed[: min(len(processed), 50)]])
    vmin, vmax = robust_limits(sample)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    writer = imageio.get_writer(
        out_path,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=16,
        ffmpeg_params=["-pix_fmt", "yuv420p", "-movflags", "+faststart"],
    )

    try:
        for i, frame in enumerate(tqdm(processed, desc=f"Rendering {out_path.name}")):
            a = background_subtract(frame)
            x = np.clip((a - vmin) / (vmax - vmin + 1e-12), 0, 1)
            x = np.arcsinh(10 * x) / np.arcsinh(10)
            gray = (255 * x).astype(np.uint8)
            rgb = np.stack([gray, gray, gray], axis=-1)
            rgb = fit_image_to_canvas(rgb, size=size, mode="contain")
            rgb = annotate_rgb(rgb, f"{label_prefix} | frame {i+1}/{len(processed)}", height=44)
            writer.append_data(rgb)
    finally:
        writer.close()

    return out_path


def acquire_jupiter_products() -> list[JupiterProduct]:
    """Search, download, and read a small set of Jupiter FITS science products."""
    obs = query_jupiter_observations()
    print(f"Found {len(obs)} matching observation rows.")
    preview_table(obs)

    if obs is None or len(obs) == 0:
        raise RuntimeError("No matching MAST observations found. Try PROGRAM_ID=None or a larger RADIUS.")

    products: list[JupiterProduct] = []
    tried = 0
    seen_files = set()

    for obs_index, obs_row in enumerate(obs[:MAX_OBS_TO_TRY], start=1):
        obs_id = _as_str(obs_row["obs_id"] if "obs_id" in obs_row.colnames else f"obs_{obs_index}")
        print(f"\nObservation {obs_index}/{min(len(obs), MAX_OBS_TO_TRY)}: {obs_id}")

        candidates = pick_candidate_products_for_obs(obs_row)
        if candidates is None or len(candidates) == 0:
            print("  No candidate FITS science products.")
            continue

        preview_cols = [c for c in ["productFilename", "productSubGroupDescription", "size", "filters"] if c in candidates.colnames]
        if preview_cols:
            display(candidates[: min(5, len(candidates))][preview_cols])

        for pr in candidates[:MAX_PRODUCTS_TO_TRY_PER_OBS]:
            if len(products) >= MAX_PRODUCTS_TO_DOWNLOAD:
                return products
            filename = _as_str(pr["productFilename"] if "productFilename" in pr.colnames else "unknown.fits")
            if filename in seen_files:
                continue
            seen_files.add(filename)
            tried += 1

            print(f"  Trying product {tried}: {filename}")
            try:
                fits_path = download_one_product(pr, DATA_DIR)
                image, cube, label, wave = read_fits_product(fits_path)
                sg = None
                sgc = subgroup_col(pr._table) if hasattr(pr, "_table") else None
                if sgc:
                    sg = _as_str(pr[sgc])
                products.append(
                    JupiterProduct(
                        fits_path=fits_path,
                        filename=filename,
                        image=image,
                        cube=cube,
                        filter_label=label,
                        wavelength_um=wave,
                        obs_id=obs_id,
                        subgroup=sg,
                    )
                )
                print(f"    OK: shape={cube.shape}, filter={label}, wavelength={wave}, local={fits_path.name}")

                # Stop early when we have a useful multi-channel set or a movie-like cube.
                unique_labels = {p.filter_label for p in products}
                has_cube = any(p.cube is not None and len(p.cube) >= 5 for p in products)
                if len(unique_labels) >= 3 or (len(products) >= 3 and has_cube):
                    return products

            except Exception as exc:
                print(f"    Failed: {type(exc).__name__}: {exc}")
                continue

    if not products:
        raise RuntimeError("Could not download/read any usable Jupiter FITS product. Increase limits or set PROGRAM_ID=None.")
    return products



## 5. Acquire Jupiter data

This cell searches MAST and downloads selected FITS products. It needs an internet connection. If you already have a Jupiter image and set `USE_LOCAL_IMAGE_ONLY = True`, the notebook skips the archive download.


In [ ]:

if USE_LOCAL_IMAGE_ONLY:
    if not LOCAL_IMAGE_PATH:
        raise ValueError("Set LOCAL_IMAGE_PATH to an existing image file when USE_LOCAL_IMAGE_ONLY=True.")
    local_rgb = np.asarray(Image.open(LOCAL_IMAGE_PATH).convert("RGB"))
    products = []
    print("Using local image only:", LOCAL_IMAGE_PATH)
else:
    products = acquire_jupiter_products()
    print("\nDownloaded/read products:")
    for p in products:
        print(f"- {p.filename} | {p.filter_label} | wave={p.wavelength_um} | cube={None if p.cube is None else p.cube.shape}")



## 6. Build and save the false-color Jupiter image


In [ ]:

if USE_LOCAL_IMAGE_ONLY:
    # Local image mode: create a pseudo-color version from the luminance of your image.
    gray = np.asarray(Image.fromarray(local_rgb).convert("L"), dtype=np.float32)
    false_rgb = pseudo_false_color_single_band(gray)
    color_mode = "Local-image pseudo false color"
else:
    false_rgb, color_mode = make_false_color_from_products(products)

annotated_false_rgb = annotate_rgb(false_rgb, f"Jupiter false color | {color_mode}", height=48)
save_rgb(annotated_false_rgb, FALSE_COLOR_PATH)

print(color_mode)
print("Saved:", FALSE_COLOR_PATH.resolve())
display(Image.fromarray(annotated_false_rgb))



## 7. Export a 1920 × 1080 still frame


In [ ]:

fhd_rgb = fit_image_to_canvas(annotated_false_rgb, size=FHD_SIZE, mode=VIDEO_FIT_MODE)
save_rgb(fhd_rgb, FALSE_COLOR_FHD_PATH)
print("Saved:", FALSE_COLOR_FHD_PATH.resolve())
display(Image.fromarray(fhd_rgb))



## 8. Render a Full HD MP4 clip from the false-color image

This creates a 1920 × 1080 video from the still false-color image using a slow zoom/pan effect.


In [ ]:

render_fhd_clip_from_image(
    annotated_false_rgb,
    out_path=STILL_CLIP_PATH,
    seconds=STILL_CLIP_SECONDS,
    fps=FPS,
    size=FHD_SIZE,
    fit_mode=VIDEO_FIT_MODE,
    zoom_start=VIDEO_ZOOM_START,
    zoom_end=VIDEO_ZOOM_END,
    label="Jupiter false-color FHD clip",
)
print("Saved:", STILL_CLIP_PATH.resolve())
display(Video(str(STILL_CLIP_PATH), embed=False, width=960))



## 9. Optional: render a Full HD video from a real frame cube

If the downloaded FITS product contains multiple integrations/frames, this cell creates a true frame-sequence MP4 at 1920 × 1080. If the data are only still images, it will skip this step.


In [ ]:

if not USE_LOCAL_IMAGE_ONLY:
    movie_candidates = [p for p in products if p.cube is not None and len(p.cube) >= 2]
    if movie_candidates:
        # Pick the product with the most frames.
        best_movie = max(movie_candidates, key=lambda p: len(p.cube))
        print(f"Rendering frame-sequence video from {best_movie.filename} with {len(best_movie.cube)} frames.")
        render_fhd_video_from_cube(
            best_movie.cube,
            out_path=FRAME_SEQUENCE_PATH,
            fps=FRAME_SEQUENCE_FPS,
            size=FHD_SIZE,
            label_prefix=f"Jupiter | {best_movie.filter_label}",
        )
        print("Saved:", FRAME_SEQUENCE_PATH.resolve())
        display(Video(str(FRAME_SEQUENCE_PATH), embed=False, width=960))
    else:
        print("No multi-frame FITS cube found. The still-image FHD clip has already been created.")
else:
    print("Local-image-only mode: frame-cube video skipped.")



## 10. Output summary


In [ ]:

print("Main outputs:")
for path in [FALSE_COLOR_PATH, FALSE_COLOR_FHD_PATH, STILL_CLIP_PATH, FRAME_SEQUENCE_PATH]:
    if path.exists():
        print("✓", path.resolve())
    else:
        print("- not created:", path.resolve())
